# Notebook 02 - Análise STRIDE

**Tech Challenge - Fase 5 - Hackaton FIAP**
**Modelagem de Ameaças utilizando IA (STRIDE)**

Este notebook executa os Passos 3 e 4 do pipeline:
- **Passo 3**: Análise STRIDE por componente via Claude API
- **Passo 4**: Geração de relatórios (JSON estruturado + PDF formatado)

**GPU necessária**: Nenhuma (apenas chamadas API)

## 1. Setup e Dependências

In [ ]:
# Instalar dependências
!pip install -q anthropic fpdf2

In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime

import anthropic
from fpdf import FPDF

print("Dependências carregadas com sucesso!")

## 2. Montar Google Drive e Configurar

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Caminhos
DRIVE_BASE = Path('/content/drive/MyDrive/hackaton-stride')
DETECTIONS_DIR = DRIVE_BASE / 'outputs' / 'detections'
REPORTS_DIR = DRIVE_BASE / 'outputs' / 'reports'
IMAGES_DIR = DRIVE_BASE / 'test_images'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# API Claude
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print(f"Detecções disponíveis: {list(DETECTIONS_DIR.glob('*.json'))}")

In [ ]:
# Selecionar arquitetura (1 = AWS, 2 = Azure)
ARCH_INDEX = 1

# Carregar JSON de componentes detectados
detections_file = DETECTIONS_DIR / f'componentes_arquitetura_{ARCH_INDEX}.json'
with open(detections_file, 'r', encoding='utf-8') as f:
    detection_data = json.load(f)

provider = detection_data['provider']
components = detection_data['components']

print(f"Arquitetura: {ARCH_INDEX} ({provider})")
print(f"Componentes carregados: {len(components)}")
for c in components:
    print(f"  - {c['name']} ({c['class']})")

## 3. Passo 3 — Análise STRIDE por Componente

In [ ]:
def build_stride_prompt(component: dict, all_components: list, provider: str) -> str:
    """
    Constrói o prompt STRIDE contextualizado para um componente.
    Inclui contexto da arquitetura completa.
    """
    # Lista de todos os componentes para contexto
    arch_context = "\n".join(
        f"  - {c['name']} ({c['class']})"
        for c in all_components
    )

    prompt = f"""Você é um especialista em segurança de software e modelagem de ameaças.

Analise o componente abaixo usando a metodologia STRIDE. O componente faz parte de uma arquitetura {provider}.

## Componente a analisar
- Nome: {component['name']}
- Classe: {component['class']}
- Provedor: {provider}

## Contexto da arquitetura completa
Outros componentes presentes:
{arch_context}

## Instruções

Para cada categoria STRIDE, forneça:
1. Descrição da ameaça específica para este componente
2. Nível de risco: "Alto", "Médio" ou "Baixo"
3. Contramedida específica do provedor ({provider})
4. Referência a boas práticas ({"AWS Well-Architected Framework" if provider == "AWS" else "Azure Security Benchmark"})

Responda APENAS em JSON válido com este formato exato:
{{
  "spoofing": {{
    "threat": "Descrição da ameaça de spoofing",
    "risk": "Alto|Médio|Baixo",
    "countermeasure": "Contramedida específica",
    "reference": "Referência a boas práticas"
  }},
  "tampering": {{
    "threat": "...",
    "risk": "...",
    "countermeasure": "...",
    "reference": "..."
  }},
  "repudiation": {{
    "threat": "...",
    "risk": "...",
    "countermeasure": "...",
    "reference": "..."
  }},
  "information_disclosure": {{
    "threat": "...",
    "risk": "...",
    "countermeasure": "...",
    "reference": "..."
  }},
  "denial_of_service": {{
    "threat": "...",
    "risk": "...",
    "countermeasure": "...",
    "reference": "..."
  }},
  "elevation_of_privilege": {{
    "threat": "...",
    "risk": "...",
    "countermeasure": "...",
    "reference": "..."
  }}
}}"""

    return prompt


def analyze_stride(component: dict, all_components: list, provider: str) -> dict:
    """
    Executa análise STRIDE para um componente usando Claude API.
    """
    prompt = build_stride_prompt(component, all_components, provider)

    response = client.messages.create(
        model='claude-sonnet-4-6-20250514',
        max_tokens=2000,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )

    text = response.content[0].text.strip()

    # Extrair JSON
    try:
        start = text.find('{')
        end = text.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
    except json.JSONDecodeError as e:
        print(f"  Erro ao parsear JSON para {component['name']}: {e}")

    return {}

In [ ]:
# Executar análise STRIDE para todos os componentes
print(f"Iniciando análise STRIDE para {len(components)} componentes...\n")

stride_results = []
t0 = time.time()

for idx, component in enumerate(components):
    print(f"[{idx+1}/{len(components)}] Analisando: {component['name']} ({component['class']})...")

    threats = analyze_stride(component, components, provider)

    result = {
        'id': component['id'],
        'name': component['name'],
        'class': component['class'],
        'bbox': component['bbox'],
        'provider': provider,
        'threats': threats
    }
    stride_results.append(result)

    # Resumo rápido
    risks = [threats[cat].get('risk', '?') for cat in threats if isinstance(threats[cat], dict)]
    high_count = risks.count('Alto')
    print(f"  -> {len(threats)} categorias analisadas, {high_count} riscos altos")

    # Rate limiting
    time.sleep(1)

t_stride = time.time() - t0
print(f"\nAnálise STRIDE concluída em {t_stride:.2f}s")

## 4. Passo 4a — Gerar Relatório JSON

In [ ]:
def generate_summary(stride_results: list) -> dict:
    """
    Gera resumo executivo da análise STRIDE.
    """
    total_threats = 0
    risk_counts = {'Alto': 0, 'Médio': 0, 'Baixo': 0}
    category_counts = {
        'spoofing': 0, 'tampering': 0, 'repudiation': 0,
        'information_disclosure': 0, 'denial_of_service': 0,
        'elevation_of_privilege': 0
    }
    high_risk_components = []

    for result in stride_results:
        threats = result.get('threats', {})
        component_high = 0
        for cat, details in threats.items():
            if isinstance(details, dict):
                total_threats += 1
                risk = details.get('risk', 'Baixo')
                if risk in risk_counts:
                    risk_counts[risk] += 1
                if risk == 'Alto':
                    component_high += 1
                if cat in category_counts:
                    category_counts[cat] += 1

        if component_high > 0:
            high_risk_components.append({
                'name': result['name'],
                'class': result['class'],
                'high_risk_count': component_high
            })

    return {
        'total_components': len(stride_results),
        'total_threats': total_threats,
        'risk_distribution': risk_counts,
        'threats_by_category': category_counts,
        'high_risk_components': sorted(
            high_risk_components,
            key=lambda x: x['high_risk_count'],
            reverse=True
        )
    }

In [ ]:
# Gerar resumo e salvar JSON completo
summary = generate_summary(stride_results)

report_data = {
    'metadata': {
        'project': 'Tech Challenge - Fase 5 - Hackaton FIAP',
        'methodology': 'STRIDE',
        'generated_at': datetime.now().isoformat(),
        'image': detection_data['image'],
        'provider': provider
    },
    'components': stride_results,
    'summary': summary,
    'timing': {
        'stride_analysis_s': round(t_stride, 2),
        'detection_timing': detection_data.get('timing', {})
    }
}

# Salvar JSON
json_path = REPORTS_DIR / f'stride_arquitetura_{ARCH_INDEX}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)

print(f"Relatório JSON salvo em: {json_path}")
print(f"\nResumo executivo:")
print(f"  Componentes analisados: {summary['total_components']}")
print(f"  Total de ameaças: {summary['total_threats']}")
print(f"  Riscos: Alto={summary['risk_distribution']['Alto']}, "
      f"Médio={summary['risk_distribution']['Médio']}, "
      f"Baixo={summary['risk_distribution']['Baixo']}")
if summary['high_risk_components']:
    print(f"\n  Componentes com risco alto:")
    for c in summary['high_risk_components']:
        print(f"    - {c['name']}: {c['high_risk_count']} ameaças de alto risco")

## 5. Passo 4b — Gerar Relatório PDF

In [ ]:
class STRIDEReport(FPDF):
    """Gerador de relatório PDF para análise STRIDE."""

    def __init__(self, provider: str, arch_index: int):
        super().__init__()
        self.provider = provider
        self.arch_index = arch_index
        # Registrar fonte Unicode para suportar caracteres especiais
        self.add_font('DejaVu', '', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', uni=True)
        self.add_font('DejaVu', 'B', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', uni=True)

    def header(self):
        self.set_font('DejaVu', 'B', 10)
        self.set_text_color(100, 100, 100)
        self.cell(0, 8, f'Relatório STRIDE - Arquitetura {self.arch_index} ({self.provider})', 0, 1, 'R')
        self.line(10, 18, 200, 18)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('DejaVu', '', 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10, f'Página {self.page_no()}/{{nb}}', 0, 0, 'C')

    def add_cover(self):
        """Página de capa."""
        self.add_page()
        self.ln(60)
        self.set_font('DejaVu', 'B', 28)
        self.set_text_color(30, 30, 30)
        self.cell(0, 15, 'Relatório de Modelagem', 0, 1, 'C')
        self.cell(0, 15, 'de Ameaças - STRIDE', 0, 1, 'C')
        self.ln(10)
        self.set_font('DejaVu', '', 16)
        self.set_text_color(80, 80, 80)
        self.cell(0, 10, f'Arquitetura {self.arch_index} - {self.provider}', 0, 1, 'C')
        self.ln(20)
        self.set_font('DejaVu', '', 12)
        self.cell(0, 8, 'Tech Challenge - Fase 5 - Hackaton', 0, 1, 'C')
        self.cell(0, 8, 'FIAP - Pós Tech - Software Security', 0, 1, 'C')
        self.ln(10)
        self.set_font('DejaVu', '', 10)
        self.cell(0, 8, f'Gerado em: {datetime.now().strftime("%d/%m/%Y %H:%M")}', 0, 1, 'C')

    def add_summary(self, summary: dict):
        """Página de resumo executivo."""
        self.add_page()
        self.set_font('DejaVu', 'B', 18)
        self.set_text_color(30, 30, 30)
        self.cell(0, 12, 'Resumo Executivo', 0, 1, 'L')
        self.ln(5)

        # Métricas gerais
        self.set_font('DejaVu', 'B', 12)
        self.cell(0, 8, 'Visão Geral', 0, 1)
        self.set_font('DejaVu', '', 11)
        self.cell(0, 7, f"Componentes analisados: {summary['total_components']}", 0, 1)
        self.cell(0, 7, f"Total de ameaças identificadas: {summary['total_threats']}", 0, 1)
        self.ln(5)

        # Distribuição de riscos
        self.set_font('DejaVu', 'B', 12)
        self.cell(0, 8, 'Distribuição de Riscos', 0, 1)
        self.set_font('DejaVu', '', 11)

        risks = summary['risk_distribution']
        colors = {'Alto': (220, 53, 69), 'Médio': (255, 193, 7), 'Baixo': (40, 167, 69)}
        for risk_level, count in risks.items():
            r, g, b = colors.get(risk_level, (100, 100, 100))
            self.set_fill_color(r, g, b)
            self.set_text_color(255, 255, 255)
            self.cell(25, 7, f' {risk_level}', 0, 0, 'C', fill=True)
            self.set_text_color(30, 30, 30)
            self.cell(30, 7, f'  {count} ameaças', 0, 1)

        self.ln(5)

        # Componentes de alto risco
        if summary['high_risk_components']:
            self.set_font('DejaVu', 'B', 12)
            self.cell(0, 8, 'Componentes com Risco Alto', 0, 1)
            self.set_font('DejaVu', '', 10)
            for comp in summary['high_risk_components']:
                self.set_text_color(220, 53, 69)
                self.cell(5, 6, '!', 0, 0)
                self.set_text_color(30, 30, 30)
                self.cell(0, 6,
                    f"{comp['name']} - {comp['high_risk_count']} ameaças de alto risco",
                    0, 1)

    def add_component_table(self, components: list):
        """Tabela de componentes detectados."""
        self.add_page()
        self.set_font('DejaVu', 'B', 18)
        self.set_text_color(30, 30, 30)
        self.cell(0, 12, 'Componentes Detectados', 0, 1)
        self.ln(5)

        # Cabeçalho da tabela
        self.set_font('DejaVu', 'B', 9)
        self.set_fill_color(50, 50, 50)
        self.set_text_color(255, 255, 255)
        self.cell(10, 8, '#', 1, 0, 'C', fill=True)
        self.cell(60, 8, 'Componente', 1, 0, 'C', fill=True)
        self.cell(40, 8, 'Classe', 1, 0, 'C', fill=True)
        self.cell(30, 8, 'Provedor', 1, 0, 'C', fill=True)
        self.cell(50, 8, 'Bounding Box', 1, 1, 'C', fill=True)

        # Linhas da tabela
        self.set_font('DejaVu', '', 8)
        self.set_text_color(30, 30, 30)
        for idx, comp in enumerate(components):
            fill = idx % 2 == 0
            if fill:
                self.set_fill_color(245, 245, 245)
            self.cell(10, 7, str(idx + 1), 1, 0, 'C', fill=fill)
            self.cell(60, 7, comp['name'][:30], 1, 0, 'L', fill=fill)
            self.cell(40, 7, comp['class'], 1, 0, 'C', fill=fill)
            self.cell(30, 7, comp['provider'], 1, 0, 'C', fill=fill)
            bbox_str = str(comp.get('bbox', []))
            self.cell(50, 7, bbox_str[:25], 1, 1, 'C', fill=fill)

    def add_stride_detail(self, result: dict):
        """Análise STRIDE detalhada para um componente."""
        self.add_page()

        # Título do componente
        self.set_font('DejaVu', 'B', 14)
        self.set_text_color(30, 30, 30)
        self.cell(0, 10, f"{result['name']}", 0, 1)
        self.set_font('DejaVu', '', 10)
        self.set_text_color(100, 100, 100)
        self.cell(0, 6, f"Classe: {result['class']} | Provedor: {result['provider']}", 0, 1)
        self.ln(3)

        # Mapeamento de nomes de categorias
        category_names = {
            'spoofing': 'S - Spoofing (Falsificação)',
            'tampering': 'T - Tampering (Adulteração)',
            'repudiation': 'R - Repudiation (Negação)',
            'information_disclosure': 'I - Information Disclosure (Vazamento)',
            'denial_of_service': 'D - Denial of Service (Negação de Serviço)',
            'elevation_of_privilege': 'E - Elevation of Privilege (Elevação)'
        }

        risk_colors = {
            'Alto': (220, 53, 69),
            'Médio': (255, 193, 7),
            'Baixo': (40, 167, 69)
        }

        threats = result.get('threats', {})
        for cat_key, cat_name in category_names.items():
            if cat_key not in threats or not isinstance(threats[cat_key], dict):
                continue

            detail = threats[cat_key]

            # Verificar se precisa nova página
            if self.get_y() > 240:
                self.add_page()

            # Categoria
            self.set_font('DejaVu', 'B', 10)
            self.set_text_color(30, 30, 30)
            risk = detail.get('risk', 'Baixo')
            r, g, b = risk_colors.get(risk, (100, 100, 100))

            self.cell(140, 7, cat_name, 0, 0)
            self.set_fill_color(r, g, b)
            self.set_text_color(255, 255, 255)
            self.cell(25, 7, f' {risk} ', 0, 1, 'C', fill=True)
            self.set_text_color(30, 30, 30)

            # Detalhes
            self.set_font('DejaVu', '', 9)
            threat_text = detail.get('threat', 'N/A')
            self.multi_cell(0, 5, f"Ameaça: {threat_text}")

            countermeasure = detail.get('countermeasure', 'N/A')
            self.set_font('DejaVu', 'B', 9)
            self.cell(25, 5, 'Contramedida: ', 0, 0)
            self.set_font('DejaVu', '', 9)
            self.multi_cell(0, 5, countermeasure)

            reference = detail.get('reference', 'N/A')
            self.set_font('DejaVu', '', 8)
            self.set_text_color(100, 100, 100)
            self.multi_cell(0, 4, f"Ref: {reference}")
            self.set_text_color(30, 30, 30)
            self.ln(3)

In [ ]:
# Gerar PDF
print("Gerando relatório PDF...")

pdf = STRIDEReport(provider, ARCH_INDEX)
pdf.alias_nb_pages()

# 1. Capa
pdf.add_cover()

# 2. Resumo executivo
pdf.add_summary(summary)

# 3. Tabela de componentes
pdf.add_component_table(stride_results)

# 4. Análise STRIDE detalhada por componente
for result in stride_results:
    pdf.add_stride_detail(result)

# Salvar PDF
pdf_path = REPORTS_DIR / f'stride_arquitetura_{ARCH_INDEX}.pdf'
pdf.output(str(pdf_path))

print(f"Relatório PDF salvo em: {pdf_path}")
print(f"Páginas: {pdf.page_no()}")

## 6. Resumo da Execução

In [ ]:
print("=" * 60)
print(f"RESUMO - Análise STRIDE")
print("=" * 60)
print(f"Arquitetura: {ARCH_INDEX} ({provider})")
print(f"Componentes analisados: {summary['total_components']}")
print(f"Total de ameaças: {summary['total_threats']}")
print(f"")
print(f"Distribuição de riscos:")
print(f"  Alto:  {summary['risk_distribution']['Alto']}")
print(f"  Médio: {summary['risk_distribution']['Médio']}")
print(f"  Baixo: {summary['risk_distribution']['Baixo']}")
print(f"")
print(f"Ameaças por categoria:")
for cat, count in summary['threats_by_category'].items():
    print(f"  {cat}: {count}")
print(f"")
print(f"Tempo análise STRIDE: {t_stride:.2f}s")
print(f"")
print(f"Arquivos gerados:")
print(f"  JSON: {json_path}")
print(f"  PDF:  {pdf_path}")
print("=" * 60)